# YOLO Foundations Part 2: Real-World Solutions
**Format:** Station-based Exercises + Group Capstone

In this lab, we'll explore **Ultralytics Solutions** — high-level wrappers that transform raw detections into actionable business insights.


## Learning Objectives
- Apply region-based counting with custom polygons
- Measure speed and distance between tracked objects
- Generate analytics charts and heatmaps from video data
- Implement privacy-compliant object blurring
- Combine multiple solutions into a complete pipeline

---

## Video Sources used in the lab (you have to use videos from the given alternatives)


| Source | File | Use Case |
|--------|------|----------|
| **Local** | `Pull_ups.mp4` | Workout monitoring |
| **Local** | `parking_slots.mp4` | Parking management |
| **Local** | `Cars.mp4` | Sample traffic video |


---

## 🎥 Alternative Open-Source Video Sources

The clips above come from the course's GitHub repo, but if you want more variety (or your own footage to test generalization), here are free, open-license stock video sites you can pull from instead. All allow free download; check each clip's page for the exact license terms (most are royalty-free / no-attribution-required, but confirm before any commercial use).

**Traffic / intersection / vehicle counting (for Exercises 1.1, 1.3, 2.1, 2.2, 3.2, 4.2):**
- [Pexels — Traffic videos](https://www.pexels.com/search/videos/traffic/) — e.g. [Aerial view of urban intersection traffic](https://www.pexels.com/video/aerial-view-of-urban-intersection-traffic-35239545/), [Crossroads (top-down highway)](https://www.pexels.com/video/crossroads-17907374/)
- [Pixabay — Car traffic videos](https://pixabay.com/videos/search/car%20traffic/)
- [Mixkit — Free traffic videos](https://mixkit.co/free-stock-video/traffic/)
- [Videezy — Free traffic videos](https://www.videezy.com/free-video/traffic)

**Parking lots / drone footage (for Exercise 1.2 and the Capstone):**
- [Pexels — Aerial footage of a parking lot](https://www.pexels.com/video/aerial-footage-of-a-parking-lot-5587732/)
- [Pexels — Drone footage of cars parked in a mall parking lot](https://www.pexels.com/video/drone-footage-of-cars-parked-and-driving-in-parking-lot-of-the-mall-5607778/)
- [Pixabay — Drone videos](https://pixabay.com/videos/search/drone/)

**Workout / pose monitoring (for Exercise 4.1):**

- [Pexels — workout videos](https://www.pexels.com/video/jumping-sport-strong-fitness-4260553/)

**Tip:** download an MP4 from any of these, drop it in your working directory (or update the URL-download cells below to point at it), and re-run the exercise. Since camera angle, resolution, and lighting differ from the course videos, you'll likely need to re-tune your `region` coordinates or `meter_per_pixel` value — that's a great way to test whether your solution generalizes rather than just matching the original numbers.


---

## 1. Environment Setup

In [1]:
# Install Ultralytics and import libraries
import sys
%pip install -q ultralytics opencv-python pandas matplotlib requests sahi
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
from ultralytics import YOLO, solutions
from IPython.display import Video

def download_video(url, filename):
    """Download a video from `url` and save it locally as `filename` (skips if it already exists)."""
    if os.path.exists(filename):
        print(f"✓ Already have: {filename}")
        return
    r = requests.get(url, stream=True)
    if r.status_code == 200:
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"✓ Downloaded: {filename}")
    else:
        print(f"✗ Failed to download {filename}. Status code: {r.status_code}")

# TODO: Pick an open-source video for each use case from the "Alternative Open-Source Video
# Sources" section above (or bring your own), then paste its direct download URL below.
# Tip: on Pexels/Pixabay, right-click the download button (or the video itself) and
# "Copy link address" to get a direct file URL rather than a webpage URL.
VIDEO_URLS = {
    "Pull_ups.mp4": None,        # TODO: workout / pose video URL
    "parking_slots.mp4": None,  # TODO: parking lot / drone video URL
    "Cars.mp4": None,            # TODO: traffic / intersection video URL
}

for filename, url in VIDEO_URLS.items():
    if url:
        download_video(url, filename)
    else:
        print(f"⚠️ No URL set for {filename} yet — fill in VIDEO_URLS above, or place the file "
              f"in this folder yourself.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Downloaded: Pull_ups.mp4
✓ Downloaded: parking_slots.mp4
✓ Downloaded: Cars.mp4


---

# Station 1: Region & Boundary Analytics (40 min)

## 📍 Exercise 1.1: Custom Counting Region (15 min)

**Task:** Count vehicles crossing a specific lane using a custom polygon region. Compare results between a line and a polygon.

**Video:** Urban traffic intersection

In [2]:
video_path = "Cars.mp4"

# Make sure you've downloaded a traffic/intersection video for "Cars.mp4" via the VIDEO_URLS
# setup above (or placed your own file at this path) before running this cell.
assert os.path.exists(video_path), f"{video_path} not found — download it first (see setup cell)."

# Display the video
Video(video_path, width=640, height=360, embed=True)


In [ ]:
# 🚗 Exercise 1.1: Object Counting with Custom Region
# TODO: Fill in the pieces marked TODO below.

import cv2
import os
from ultralytics import YOLO, solutions
from IPython.display import Video

# 1. Setup
temp_output = "temp_raw.mp4"
final_output = "lane_counting_5fps.mp4"
model = YOLO("weights/yolo26n.pt")

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. Sampling Logic
TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))
print(f"Original FPS: {original_fps} | Sampling every {skip_rate} frames")

# 3. TODO: Define a polygon region over the lane you want to monitor.
#    A polygon needs at least 3 (x, y) points, e.g.:
#    lane_region = [(int(w*0.4), int(h*0.6)), (int(w*0.6), int(h*0.6)),
#                   (int(w*0.9), int(h*0.9)), (int(w*0.1), int(h*0.9))]
lane_region = None  # TODO

# TODO: Initialize the ObjectCounter solution.
#    Hint: solutions.ObjectCounter(region=lane_region, model=model, classes=[2, 3, 5, 7], show=False)
#    classes 2,3,5,7 correspond to car, motorcycle, bus, truck in the COCO dataset.
counter = None  # TODO

# 4. Video Writer set to 5 FPS
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    # Only process frames that match our 5 FPS target
    if frame_idx % skip_rate == 0:
        # TODO: run the counter on `im0` (e.g. results = counter(im0))
        # TODO: write the annotated frame to `writer` (e.g. writer.write(results.plot_im))
        pass

    frame_idx += 1
    if frame_idx > 300: break # Stop after ~10 seconds of source video

cap.release()
writer.release()

# 5. Notebook Display Fix
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

# TODO: print the IN/OUT counts, e.g. counter.in_count and counter.out_count
print(f"📊 Processed {frame_idx // skip_rate} frames.")  # TODO: append IN/OUT counts

Video(final_output, embed=True, width=800)


### 🏆 Challenge (Optional)
Try different region shapes:
- A horizontal line across half the frame
- A rectangle covering only the left lane
- Which gives more accurate counts?

---

## 🅿️ Exercise 1.2: Parking Lot Management (15 min)

**Task:** Define parking spots and track occupancy over time. Log results to CSV.

**Video:** Drone footage of mall parking lot

In [4]:
video_path = "parking_slots.mp4"

# Make sure you've downloaded a parking-lot / drone video for "parking_slots.mp4" via the
# VIDEO_URLS setup above (or placed your own file at this path) before running this cell.
assert os.path.exists(video_path), f"{video_path} not found — download it first (see setup cell)."


In [ ]:
# 🅿️ Exercise 1.2: Parking Lot Management
# TODO: Implement parking occupancy tracking and log results to a CSV file.
#
# Steps to guide you:
# 1. Import solutions and set up video_path = "parking_slots.mp4" (already downloaded above).
# 2. Use Ultralytics' "Parking slots annotator" tool (offline, run once) to draw parking-space
#    polygons on a still frame from the video and export them as a JSON file
#    (see: https://docs.ultralytics.com/guides/parking-management/).
# 3. Initialize the solution:
#      parking = solutions.ParkingManagement(
#          model="weights/yolo26n.pt",
#          json_file="path/to/your_parking_regions.json",
#      )
# 4. Loop over video frames (consider sampling at a reduced FPS like the other exercises),
#    call `parking(frame)`, and write the annotated frame to an output video.
# 5. Log per-frame occupancy counts (occupied vs. free spaces) to a list of dicts, then
#    save them with pandas to a CSV, e.g. `parking_occupancy.csv`.

import cv2
import os
import pandas as pd
from ultralytics import YOLO, solutions
from IPython.display import Video

video_path = "parking_slots.mp4"

# TODO: annotate parking regions and point to your JSON file
json_file = None  # TODO

# TODO: initialize solutions.ParkingManagement(...)
parking = None  # TODO

occupancy_log = []  # TODO: append {"frame": frame_idx, "occupied": ..., "free": ...} each processed frame

# TODO: open the video, loop through frames, run `parking(frame)`, write annotated output

# TODO: save occupancy_log to CSV with pandas
# pd.DataFrame(occupancy_log).to_csv("parking_occupancy.csv", index=False)


### 🏆 Challenge
Add a real-time alert when occupancy exceeds 75%

---

## 📋 Exercise 1.3: Queue Alert System (10 min)

**Task:** Monitor a region and trigger alert when queue exceeds threshold.

In [ ]:
# 📋 Exercise 1.3: Queue Alert System
# TODO: Fill in the pieces marked TODO below.

import cv2
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO, solutions
from shapely.geometry import Point, Polygon
import numpy as np

# 1. Setup
temp_output = "sahi_queue_raw.mp4"
final_output = "sahi_queue_5fps.mp4"
video_path = "Cars.mp4"

# Load Model using SAHI's AutoDetectionModel
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="weights/yolo26n.pt",  # Use your specific weights
    confidence_threshold=0.3,
    device="cpu",  # or "cuda"
)

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. Sampling Logic
TARGET_FPS = 3
skip_rate = max(1, int(original_fps / TARGET_FPS))

# 3. TODO: Define your queue region as a Polygon (shapely)
#    e.g. queue_points = [(int(w*0.3), int(h*0.5)), (int(w*0.7), int(h*0.5)),
#                         (int(w*0.9), int(h*0.9)), (int(w*0.1), int(h*0.9))]
queue_points = None  # TODO
queue_polygon = None  # TODO: Polygon(queue_points)

# TODO: pick a threshold for the alert, e.g. QUEUE_THRESHOLD = 5

# 4. Processing Loop
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # Perform Sliced Inference (SAHI) -- helps detect small/far objects
        result = get_sliced_prediction(
            frame,
            detection_model,
            slice_height=640,
            slice_width=640,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2
        )

        occupancy_count = 0
        annotated_frame = frame.copy()

        # TODO: for each object_prediction in result.object_prediction_list:
        #   - get its bounding box: bbox = object_prediction.bbox.to_xyxy()
        #   - compute its center point
        #   - check whether the center point lies inside `queue_polygon`
        #     (hint: queue_polygon.contains(Point(cx, cy)))
        #   - if inside, increment occupancy_count and draw a marker on annotated_frame

        # TODO: if occupancy_count exceeds your threshold, draw/print an alert

        writer.write(annotated_frame)

    frame_idx += 1
    if frame_idx > 300: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


---

# Station 2: Kinematic & Spatial Measurement (35 min)

## 🚀 Exercise 2.1: Speed Estimation (20 min)

**Task:** Estimate vehicle speeds by calibrating `meter_per_pixel`. Identify the fastest vehicle.

**Hint:** Start with `meter_per_pixel=0.05` and adjust based on visual reference.

In [ ]:
# 🚀 Exercise 2.1: Speed Estimation
# TODO: Fill in the pieces marked TODO below.

video_path = "Cars.mp4"
temp_output = "speed_raw.mp4"
final_output = "speed_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Calibrate this. Start at 0.05 and adjust based on a visual reference
# (e.g. known lane width or vehicle length in pixels vs. real-world meters).
METER_PER_PIXEL = None  # TODO

# TODO: Initialize solutions.SpeedEstimator
#   Hint: solutions.SpeedEstimator(show=False, model=model, fps=TARGET_FPS,
#                                   meter_per_pixel=METER_PER_PIXEL, classes=[2, 3, 5, 7])
speed_est = None  # TODO

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

speed_readings = []
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run speed_est on im0 and write the annotated frame
        # results = speed_est(im0)
        # writer.write(results.plot_im)

        # TODO: collect per-track speed readings into `speed_readings`
        # if hasattr(results, 'speed') and results.speed is not None:
        #     for track_id, speed in results.speed.items():
        #         if speed > 0:
        #             speed_readings.append({'frame': frame_idx, 'track_id': track_id, 'speed_kmh': speed})
        pass

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

# Summary stats
if speed_readings:
    df = pd.DataFrame(speed_readings)
    print(f"\n📊 Avg speed: {df['speed_kmh'].mean():.1f} km/h | Max speed: {df['speed_kmh'].max():.1f} km/h")
else:
    print("⚠️ No speed data captured")

from IPython.display import Video
Video(final_output, embed=True, width=800)


### 🏆 Challenge
Adjust `meter_per_pixel` to get more realistic speeds. What value works best?

---

## 📏 Exercise 2.2: Distance Calculation (15 min)

**Task:** Calculate distance between two tracked vehicles. Trigger warning if too close.

In [ ]:
# 📏 Exercise 2.2: Distance Calculation
# TODO: Fill in the pieces marked TODO below.

video_path = "Cars.mp4"
temp_output = "distance_raw.mp4"
final_output = "distance_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Initialize solutions.DistanceCalculation
#   Hint: solutions.DistanceCalculation(show=False, model=model, classes=[2, 3, 5, 7])
dist_calc = None  # TODO

# TODO: pick a real-world distance threshold (in your chosen unit) below which
#       you want to trigger a "too close" warning.

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run dist_calc on im0 and write the annotated frame
        # results = dist_calc(im0)
        # writer.write(results.plot_im)

        # TODO: inspect the computed distance between tracked pairs and print/draw
        #       a warning if it drops below your threshold
        pass

    frame_idx += 1
    if frame_idx > 150: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


---

# Station 3: Privacy, Analytics & Visualization (35 min)

## 🔒 Exercise 3.1: Object Blurring (10 min)

**Task:** Blur detected persons for privacy compliance.

**Challenge:** Try to blur only faces using pose keypoints (advanced).

In [ ]:
video_path = "parking_slots.mp4"

# Make sure you've downloaded a parking-lot / drone video for "parking_slots.mp4" via the
# VIDEO_URLS setup above (or placed your own file at this path) before running this cell.
assert os.path.exists(video_path), f"{video_path} not found — download it first (see setup cell)."


In [ ]:
# 🔒 Exercise 3.1: Object Blurring
# TODO: Implement privacy-compliant blurring of detected people.

video_path = "parking_slots.mp4"
temp_output = "blur_raw.mp4"
final_output = "blur_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Initialize solutions.ObjectBlurrer
#   Hint: solutions.ObjectBlurrer(show=False, model=model, classes=[0])  # class 0 = person
blurrer = None  # TODO

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run blurrer on im0 and write the annotated (blurred) frame
        pass

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


### 🏆 Advanced Challenge: Face-Only Blur
Use YOLO Pose (`yolov8n-pose.pt`) and blur only keypoints 0-4 (face)

---

## 📈 Exercise 3.2: Analytics Dashboard (15 min)

**Task:** Generate line/bar/pie charts showing object class distribution over time.

In [ ]:
# 📈 Exercise 3.2: Analytics Dashboard
# TODO: Fill in the pieces marked TODO below.

video_path = "Cars.mp4"
temp_output = "analytics_raw.mp4"
final_output = "analytics_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Initialize solutions.Analytics
#   Hint: solutions.Analytics(show=False, analytics_type="line", model=model, classes=[2, 3, 5, 7])
#   Try changing analytics_type to "bar", "pie", or "area" for the challenge below.
analytics = None  # TODO

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run analytics on im0 (pass frame_count=frame_idx) and write the annotated frame
        # results = analytics(im0, frame_count=frame_idx)
        # writer.write(results.plot_im)
        pass

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


### 🏆 Challenge
Create all 4 chart types (line, bar, pie, area) and identify peak hours

---

## 🔥 Exercise 3.3: Heatmap Visualization (10 min)

**Task:** Generate a heatmap showing movement density. Compare colormaps.

In [ ]:
video_path = "parking_slots.mp4"

# Make sure you've downloaded a parking-lot / drone video for "parking_slots.mp4" via the
# VIDEO_URLS setup above (or placed your own file at this path) before running this cell.
assert os.path.exists(video_path), f"{video_path} not found — download it first (see setup cell)."


In [ ]:
# 🔥 Exercise 3.3: Heatmap Visualization
# TODO: Implement a movement-density heatmap and compare colormaps.

video_path = "parking_slots.mp4"
temp_output = "heatmap_raw.mp4"
final_output = "heatmap_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Initialize solutions.Heatmap
#   Hint: solutions.Heatmap(show=False, model=model, colormap=cv2.COLORMAP_JET, classes=[2, 3, 5, 7])
#   Try a different colormap constant (e.g. cv2.COLORMAP_HOT, cv2.COLORMAP_TURBO) for the challenge.
heatmap = None  # TODO

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run heatmap on im0 and write the annotated frame
        pass

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


---

# Station 4: Specialized Heads (30 min)

## 🏋️ Exercise 4.1: Workout Monitoring (15 min)

**Task:** Count pushups using YOLO Pose. Modify for squats or jumping jacks.

**Video:** Person doing pull-ups

In [ ]:
# 🏋️ Exercise 4.1: Workout Monitoring
# TODO: Fill in the pieces marked TODO below.

video_path = "Pull_ups.mp4"
temp_output = "gym_raw.mp4"
final_output = "gym_5fps.mp4"

model = YOLO("yolov8n-pose.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# TODO: Initialize solutions.AIGym with the right keypoints for counting pull-ups.
#   Hint: shoulder-elbow-wrist is kpts=[6, 8, 10] (COCO pose keypoint indices).
#   For the challenge: squats = [11, 13, 15] (hip-knee-ankle),
#                      jumping jacks = [5, 7, 9] + [6, 8, 10] (both arms).
gym = None  # TODO

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        # TODO: run gym on im0 and write the annotated frame
        pass

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

# TODO: print the rep count tracked by `gym`
print(f"💪 Rep count: N/A")  # TODO

from IPython.display import Video
Video(final_output, embed=True, width=800)


### 🏆 Challenge
Modify the keypoints for:
- **Squats:** kpts=[11, 13, 15] (hip-knee-ankle)
- **Jumping jacks:** kpts=[5, 7, 9] + [6, 8, 10] (both arms)

---

## 🎭 Exercise 4.2: Instance Segmentation Tracking (15 min)

**Task:** Track objects with pixel-perfect masks. Compare to bounding box tracking.

In [ ]:
# 🎭 Exercise 4.2: Instance Segmentation Tracking
# TODO: Fill in the pieces marked TODO below.

video_path = "Cars.mp4"

# TODO: Load a YOLO segmentation model instead of a plain detection model
#   Hint: YOLO("yolo26n-seg.pt")
model = None  # TODO

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

frame_count = 0
max_frames = 50

# TODO: track using the segmentation model with model.track(...)
#   Hint: model.track(source=video_path, stream=True, classes=[2, 3, 5, 7], persist=True)
results = None  # TODO

tracked_objects = set()

# TODO: iterate over `results`, collect unique track IDs from result.boxes.id into tracked_objects,
#       stop after `max_frames`

print(f"✓ Tracked {len(tracked_objects)} unique vehicles with instance masks")

# 🏆 Bonus: compare `tracked_objects` here to the count you got from a plain bounding-box
# tracker (e.g. model.track with a non-segmentation model) on the same video/frame range.
# Are the counts the same? Where do they differ, and why might that be?


---

# Capstone Project: Smart Retail Analytics System (60 min)

## 🎯 Group Challenge (3-4 students)

Build a complete retail analytics system that:

1. **Counts customers** entering the store (ObjectCounter)
2. **Generates heatmap** of popular aisles (Heatmap)
3. **Blurs faces** for privacy (ObjectBlurrer)
4. **Shows peak hours** with analytics chart (Analytics)
5. **Alerts** if checkout queue > 3 people (Queue alert)

### Requirements
- Use **at least 3 different solutions**
- Process video end-to-end
- Export results to CSV/JSON

### Stretch Goals
- Add speed estimation for customer walking patterns
- Create a simple dashboard visualization
- Combine multiple videos

### Deliverables
- Working notebook with all solutions integrated
- `retail_summary.csv` with counts, peak times, occupancy
- 2-minute demo per group

In [ ]:
video_path = "parking_slots.mp4"

# Make sure you've downloaded a parking-lot / drone video for "parking_slots.mp4" via the
# VIDEO_URLS setup above (or placed your own file at this path) before running this cell.
assert os.path.exists(video_path), f"{video_path} not found — download it first (see setup cell)."


In [ ]:
# 🎯 Capstone: Smart Retail Analytics System
# TODO: Build your team's pipeline here. Use at least 3 of the Solutions you practiced above:
#   - ObjectCounter   (customers entering the store)
#   - Heatmap         (popular aisles)
#   - ObjectBlurrer   (privacy for faces/people)
#   - Analytics       (peak hours chart)
#   - Queue alert     (checkout line > 3 people)
#   - SpeedEstimator  (stretch: customer walking patterns)
#
# Suggested structure:
# 1. Load your video(s) and model(s).
# 2. Initialize the 3+ solutions you're combining.
# 3. Loop over frames once, running each solution and merging their outputs into a single
#    annotated frame (or run separate passes if that's simpler for your team).
# 4. Collect the metrics each solution produces into a results dict/list per frame.
# 5. Export a summary to retail_summary.csv (counts, peak times, occupancy, etc.).
# 6. (Optional) Build a small dashboard visualization of the results.

import cv2
import os
import pandas as pd
from ultralytics import YOLO, solutions
from IPython.display import Video

video_path = "parking_slots.mp4"  # TODO: swap in your own retail/store footage if you have it

# TODO: your team's pipeline


---

## ✅ Wrap-up Questions

1. Which solution was most challenging to configure?
2. How would you adapt these solutions for a real deployment?
3. What other business problems could these solutions solve?

---

## 📚 References

- [Object Counting Docs](https://docs.ultralytics.com/guides/object-counting/)
- [Heatmaps Docs](https://docs.ultralytics.com/guides/heatmaps/)
- [Speed Estimation Docs](https://docs.ultralytics.com/guides/speed-estimation/)
- [Analytics Docs](https://docs.ultralytics.com/guides/analytics/)
- [Parking Management Docs](https://docs.ultralytics.com/guides/parking-management/)
- [Workouts Monitoring Docs](https://docs.ultralytics.com/guides/workouts-monitoring/)

---

**End of Lab**